# Retrieval Augmented Generation (RAG) dengan Pendekatan Agent
Notebook ini mendemonstrasikan dua skenario implementasi RAG berbasis Agent (ReAct) menggunakan database vektor Chroma dan model Gemini:
1. **Contoh 1**: RAG menggunakan satu berkas dokumen PDF (`pemilu 2024.pdf`).
2. **Contoh 2**: RAG menggunakan banyak dokumen dengan tipe berkas berbeda (PDF `pemilu 2024.pdf` dan TXT `demografi-jakarta.txt`).

## Contoh 1: RAG dengan Satu Dokumen (PDF)

In [5]:
# 1. Memuat variabel lingkungan
from dotenv import load_dotenv
load_dotenv()

True

In [7]:
# 2. Memuat file PDF secara manual menggunakan pypdf
from pypdf import PdfReader
from langchain_core.documents import Document

pdf_file_path = "data/pemilu 2024.pdf"
reader = PdfReader(pdf_file_path)

docs_single = []
for i, page in enumerate(reader.pages):
    text = page.extract_text()
    docs_single.append(Document(page_content=text, metadata={"source": pdf_file_path, "page": i}))

print(f"Total halaman dokumen: {len(docs_single)}")

Total halaman dokumen: 2


In [8]:
# 3. Membagi dokumen menjadi beberapa chunk/potongan teks kecil
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
splits_single = text_splitter.split_documents(docs_single)

print(f"Total Chunks : {len(splits_single)}")

Total Chunks : 5


In [9]:
# 4. Inisialisasi Model LLM dan Embeddings dari Google GenAI
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

llm = ChatGoogleGenerativeAI(
    model="gemini-3.1-flash-lite",
    temperature=0,
    max_retries=2,
)

embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
)

In [10]:
# 5. Membuat database vektor Chroma (in-memory) dan retriever
from langchain_chroma import Chroma

vectorstore_single = Chroma.from_documents(
    documents=splits_single,
    embedding=embeddings,
)

retriever_single = vectorstore_single.as_retriever(
    search_kwargs={"k": 4}
)

print("Vector Database Single Ready")

Vector Database Single Ready


In [11]:
# 6. Mendefinisikan tool pencarian dokumen tunggal
from langchain_core.tools import tool

@tool
def cari_informasi_pemilu(query: str) -> str:
    """Cari dan dapatkan informasi relevan tentang Pemilu 2024 dari dokumen resmi."""
    docs = retriever_single.invoke(query)
    return "\n\n".join(doc.page_content for doc in docs)

In [12]:
# 7. Membuat Agent untuk RAG Dokumen Tunggal
from langchain.agents import create_agent
from langchain_core.messages import HumanMessage

system_prompt_single = """
Kamu adalah asisten BPS.
Jawab pertanyaan hanya berdasarkan informasi yang diperoleh dari tool yang tersedia.
Jika jawaban tidak ditemukan, jawab: 'Maaf, saya tidak menemukan informasi tersebut pada dokumen.'
Jangan menggunakan pengetahuan di luar dokumen resmi.
Jawab maksimal 3 kalimat.
"""

agent_single = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[cari_informasi_pemilu],
    system_prompt=system_prompt_single
)

In [13]:
# 8. Menguji Agent Dokumen Tunggal
question = HumanMessage(content="kapan pemilu 2024?")
response = agent_single.invoke({"messages": [question]})

print("Pertanyaan:")
print(question.content)
print("\nJawaban Agent:")
print(response["messages"][-1].content)

Pertanyaan:
kapan pemilu 2024?

Jawaban Agent:
[{'type': 'text', 'text': 'Pemungutan suara Pemilihan Umum (Pemilu) 2024 dilaksanakan pada Rabu, 14 Februari 2024. Sementara itu, pemilihan gubernur, bupati, dan walikota disepakati digelar pada 27 November 2024.', 'extras': {'signature': 'EjQKMgERTTIP09eytLc1nmuSvi0441apw0uzRA1rlWKvGJeLTE4zNxrSQvYIIHQNc/RGvwLO'}}]


## Persist Directory

In [14]:
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_chroma import Chroma

persist_directory = "vector"

embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

print("[green]Membuat & menyimpan Vector Database...[/]")
vectorstore = Chroma.from_documents(
    documents=splits_single, # hasil dari text_splitter sebelumnya
    embedding=embeddings,
    persist_directory=persist_directory,# inilah yang bikin datanya tersimpan ke disk
)

[green]Membuat & menyimpan Vector Database...[/]


In [15]:
# Membaca dari Chroma DB yang sudah tersimpan
persist_directory = "vector"

print("[green]Load DB[/]")
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")
vectorstore = Chroma(persist_directory=persist_directory, embedding_function=embeddings)
retriever = vectorstore.as_retriever()

[green]Load DB[/]


## Stream

In [16]:
# menguji agent dengan streaming
question = HumanMessage(content="kapan pemilu 2024?")

print("Pertanyaan:")
print(question.content)
print("\nJawaban Agent (streaming):")

for chunk in agent_single.stream(
    {"messages": [question]},
    stream_mode="updates",
):
    print(chunk)
    # Setiap chunk adalah potongan hasil yang bisa langsung
    # ditampilkan ke UI chat, tanpa menunggu semua selesai.


Pertanyaan:
kapan pemilu 2024?

Jawaban Agent (streaming):
{'model': {'messages': [AIMessage(content=[], additional_kwargs={'function_call': {'name': 'cari_informasi_pemilu', 'arguments': '{"query": "kapan pemilu 2024?"}'}, '__gemini_function_call_thought_signatures__': {'3EscExcS': 'EjQKMgERTTIPMO8JD44gnhinqmC9zUFqskjuEWB289nxP8jXTzcKg5YlcZjA2BHSlSY3Gb4n'}}, response_metadata={'finish_reason': 'STOP', 'model_name': 'gemini-3.1-flash-lite', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--019f84f6-845c-7e10-9266-7c3979a94203-0', tool_calls=[{'name': 'cari_informasi_pemilu', 'args': {'query': 'kapan pemilu 2024?'}, 'id': '3EscExcS', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 132, 'output_tokens': 29, 'total_tokens': 161, 'input_token_details': {'cache_read': 0}})]}}
{'tools': {'messages': [ToolMessage(content='Sementara untuk pemilihan gubernur, bupati, dan walikota, disepakati bakal digelar \npada 27 November 2024.\nKPU berencana m

## Chunk Size

In [18]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter_kecil = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=50)
splitter_besar = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=300)

splits_kecil = splitter_kecil.split_documents(docs_single)
splits_besar = splitter_besar.split_documents(docs_single)

print(f"Chunk kecil: {len(splits_kecil)} potongan")
print(f"Chunk besar: {len(splits_besar)} potongan")
# Coba buat retriever dari masing-masing, bandingkan kualitas jawabannya

Chunk kecil: 15 potongan
Chunk besar: 3 potongan


In [24]:
## Chunk kecil
vectorstore_kecil = Chroma.from_documents(
    documents=splits_kecil,
    embedding=embeddings,
)

retriever_kecil = vectorstore_kecil.as_retriever(
    search_kwargs={"k": 4}
)
print("Vector Database Kecil Ready")

Vector Database Kecil Ready


In [25]:
## Chunk besar
vectorstore_besar = Chroma.from_documents(
    documents=splits_besar,
    embedding=embeddings,
)

retriever_besar = vectorstore_besar.as_retriever(
    search_kwargs={"k": 4}
)

print("Vector Database Besar Ready")

Vector Database Besar Ready


In [29]:
def pemilu_tool(retriever):
        @tool
        def cari_informasi_pemilu(query: str) -> str:
            """Cari dan dapatkan informasi relevan tentang Pemilu 2024 dari dokumen resmi."""
            docs = retriever.invoke(query)
            return "\n\n".join(doc.page_content for doc in docs)
        return cari_informasi_pemilu

In [34]:
system_prompt_single = """
Kamu adalah asisten BPS.
Jawab pertanyaan hanya berdasarkan informasi yang diperoleh dari tool yang tersedia.
Jika jawaban tidak ditemukan, jawab: 'Maaf, saya tidak menemukan informasi tersebut pada dokumen.'
Jangan menggunakan pengetahuan di luar dokumen resmi.
Jawab maksimal 3 kalimat.
"""

agent_kecil = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[pemilu_tool(retriever_kecil)],
    system_prompt=system_prompt_single
)

agent_besar = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[pemilu_tool(retriever_besar)],
    system_prompt=system_prompt_single
)

question = HumanMessage(content="Rangkum seluruh tahapan persiapan dan pelaksanaan Pemilu 2024 dari awal hingga pengucapan sumpah presiden beserta tanggal-tanggal pentingnya secara detail!")

In [35]:
# Menguji Agent Chunk Kecil 
response = agent_kecil.invoke({"messages": [question]})

print("Pertanyaan:")
print(question.content)
print("\nJawaban Agent:")
print(response["messages"][-1].content)

Pertanyaan:
Rangkum seluruh tahapan persiapan dan pelaksanaan Pemilu 2024 dari awal hingga pengucapan sumpah presiden beserta tanggal-tanggal pentingnya secara detail!

Jawaban Agent:
[{'type': 'text', 'text': 'Tahapan pelaksanaan Pemilu 2024 dimulai dari pemungutan suara pada 14 Februari 2024, diikuti penghitungan suara hingga 15 Februari 2024, serta rekapitulasi hasil penghitungan suara pada 15 Februari hingga 20 Maret 2024. Penetapan hasil Pemilu dilakukan tiga hari setelah pemberitahuan atau putusan Mahkamah Konstitusi. Puncak rangkaian kegiatan adalah pengucapan sumpah/janji anggota DPR dan DPD pada 1 Oktober 2024, serta pengucapan sumpah/janji Presiden dan Wakil Presiden pada 20 Oktober 2024.', 'extras': {'signature': 'EjQKMgERTTIPCrwGmuy4B1S9/77bl6K9XCz4EBoL3CIM+L6y2dibKDmvW5iyBxmYhNH5PsGQ'}}]


In [36]:
# Menguji Agent Chunk Besar 
response = agent_besar.invoke({"messages": [question]})

print("Pertanyaan:")
print(question.content)
print("\nJawaban Agent:")
print(response["messages"][-1].content)

Pertanyaan:
Rangkum seluruh tahapan persiapan dan pelaksanaan Pemilu 2024 dari awal hingga pengucapan sumpah presiden beserta tanggal-tanggal pentingnya secara detail!

Jawaban Agent:
[{'type': 'text', 'text': 'Tahapan Pemilu 2024 meliputi perencanaan program (14 Juni 2022–14 Juni 2024), pendaftaran dan verifikasi peserta Pemilu (29 Juli 2022–13 Desember 2022), serta pemungutan suara yang dilaksanakan pada 14 Februari 2024. Selain itu, terdapat tahapan pemutakhiran data pemilih, penetapan jumlah kursi dan daerah pemilihan, serta pencalonan anggota legislatif yang berlangsung sepanjang tahun 2022 hingga 2023. Maaf, saya tidak menemukan informasi mengenai jadwal spesifik pengucapan sumpah presiden pada dokumen yang tersedia.', 'extras': {'signature': 'EjQKMgERTTIPPIohce2cTPuakZiMF476gNxn/tb/kr1o9d6jGXs3D047pPh+TL5RQo+xPoyW'}}]


## Konfigurasi Retriever

In [37]:
retriever_similarity = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 4},
)

retriever_mmr = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "fetch_k": 20},
)
# fetch_k = jumlah kandidat awal sebelum disaring jadi k hasil akhir

In [38]:
agent_similarity = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[pemilu_tool(retriever_similarity)],
    system_prompt=system_prompt_single
)

agent_mmr = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[pemilu_tool(retriever_mmr)],
    system_prompt=system_prompt_single
)

question = HumanMessage(content="Rangkum seluruh tahapan persiapan dan pelaksanaan Pemilu 2024 dari awal hingga pengucapan sumpah presiden beserta tanggal-tanggal pentingnya secara detail!")

In [39]:
# Menguji Agent Similarity 
response = agent_similarity.invoke({"messages": [question]})

print("Pertanyaan:")
print(question.content)
print("\nJawaban Agent:")
print(response["messages"][-1].content)

Pertanyaan:
Rangkum seluruh tahapan persiapan dan pelaksanaan Pemilu 2024 dari awal hingga pengucapan sumpah presiden beserta tanggal-tanggal pentingnya secara detail!

Jawaban Agent:
[{'type': 'text', 'text': 'Tahapan Pemilu 2024 dimulai dari perencanaan program dan anggaran (14 Juni 2022-14 Juni 2024), pendaftaran dan penetapan peserta Pemilu, hingga pencalonan anggota legislatif dan presiden yang berakhir pada 25 November 2023. Setelah masa kampanye (28 November 2023-10 Februari 2024) dan masa tenang (11-13 Februari 2024), pemungutan suara dilaksanakan. Maaf, saya tidak menemukan informasi mengenai tanggal pengucapan sumpah presiden pada dokumen.', 'extras': {'signature': 'EjQKMgERTTIP9i/Y0zAjyKIc7dNn/ZdY6/of/WRkGEqgJkePSEW133e0t71v6IhgP+Pya5wI'}}]


In [40]:
# Menguji Agent mmr
response = agent_mmr.invoke({"messages": [question]})

print("Pertanyaan:")
print(question.content)
print("\nJawaban Agent:")
print(response["messages"][-1].content)

Pertanyaan:
Rangkum seluruh tahapan persiapan dan pelaksanaan Pemilu 2024 dari awal hingga pengucapan sumpah presiden beserta tanggal-tanggal pentingnya secara detail!

Jawaban Agent:
[{'type': 'text', 'text': 'Tahapan Pemilu 2024 dimulai dari perencanaan program pada Juni 2022 hingga pengucapan sumpah/janji Presiden dan Wakil Presiden pada 20 Oktober 2024. Tahapan krusial meliputi pendaftaran peserta, pencalonan, masa kampanye (28 November 2023 – 10 Februari 2024), hingga pemungutan suara pada 14 Februari 2024. Proses kemudian dilanjutkan dengan penghitungan, rekapitulasi, penetapan hasil, serta pelantikan anggota legislatif pada 1 Oktober 2024 sebelum pelantikan presiden.', 'extras': {'signature': 'EjQKMgERTTIPeq7VvkeyJGmQnms8w+xiTnS4vmG3ANzA2rQNyqG6bHSSOv0amUcWP4qHYw3T'}}]


## Metadata filtering

In [47]:
# 1. Menambahkan metadata kustom ('kategori') saat pemuatan dokumen
docs_dengan_metadata = []

# A. Dokumen Pemilu diberi kategori "pemilu"
for i, page in enumerate(reader.pages):
    text = page.extract_text()
    docs_dengan_metadata.append(
        Document(
            page_content=text,
            metadata={"source": pdf_file_path, "page": i, "kategori": "pemilu"}
        )
    )

# B. Dokumen Demografi diberi kategori "demografi"
with open("data/demografi-jakarta.txt", "r", encoding="utf-8") as f:
    text_demografi = f.read()
    docs_dengan_metadata.append(
        Document(
            page_content=text_demografi,
            metadata={"source": "data/demografi-jakarta.txt", "kategori": "demografi"}
        )
    )

# 2. Potong dokumen dan buat Vectorstore
splits_metadata = text_splitter.split_documents(docs_dengan_metadata)
vectorstore_meta = Chroma.from_documents(
    documents=splits_metadata,
    embedding=embeddings
)

# 3. Pencarian menggunakan Filter Metadata
# Contoh A: Filter menggunakan similarity_search langsung
hasil_pemilu = vectorstore_meta.similarity_search(
    "kapan pemilu 2024?",
    k=3,
    filter={"kategori": "pemilu"}
)

print("=== Hasil Filter Pemilu ===")
for doc in hasil_pemilu:
    print(f"Metadata: {doc.metadata} -> {doc.page_content[:80]}...\n")

# Contoh B: Membuat Retriever terfilter untuk digunakan Agent/Tool
retriever_demografi = vectorstore_meta.as_retriever(
    search_kwargs={
        "k": 3,
        "filter": {"kategori": "demografi"}
    }
)

print("=== Hasil Filter Demografi (via Retriever) ===")
docs_demografi = retriever_demografi.invoke("berapa populasi jakarta?")
for doc in docs_demografi:
    print(f"Metadata: {doc.metadata} -> {doc.page_content[:80]}...\n")

=== Hasil Filter Pemilu ===
Metadata: {'page': 0, 'source': 'data/pemilu 2024.pdf', 'kategori': 'pemilu'} -> Sementara untuk pemilihan gubernur, bupati, dan walikota, disepakati bakal digel...

Metadata: {'source': 'data/pemilu 2024.pdf', 'page': 0, 'kategori': 'pemilu'} -> # Pemilihan Umum 2024
Dokumen ini berisi Peraturan KPU no 3 tahun 2022 tentang t...

Metadata: {'source': 'data/pemilu 2024.pdf', 'page': 1, 'kategori': 'pemilu'} -> 11. Rabu, 14 Februari 2024: Pemungutan suara
12. Rabu, 14 Februari 2024 - Kamis,...

=== Hasil Filter Demografi (via Retriever) ===
Metadata: {'source': 'data/demografi-jakarta.txt', 'kategori': 'demografi'} -> # Demografi Penduduk Jakarta

Berikut adalah beberapa data demografi penduduk Ja...

Metadata: {'source': 'data/demografi-jakarta.txt', 'kategori': 'demografi'} -> Berikut ini jumlah penduduk menurut umur di Kota Jakarta Pusat pada Juni 2024 be...

Metadata: {'source': 'data/demografi-jakarta.txt', 'kategori': 'demografi'} -> Untuk di kota Jakart

## Contoh 2: RAG dengan Banyak Dokumen (Multifile - PDF & TXT)

In [9]:
# 1. Memuat beberapa file berbeda (PDF dan TXT)
import os
from pypdf import PdfReader
from langchain_core.documents import Document

docs_multi = []

# A. Memuat PDF
pdf_path = "data/pemilu 2024.pdf"
if os.path.exists(pdf_path):
    print(f"Memuat dokumen PDF: {pdf_path}")
    reader = PdfReader(pdf_path)
    for i, page in enumerate(reader.pages):
        text = page.extract_text()
        docs_multi.append(Document(page_content=text, metadata={"source": pdf_path, "page": i}))

# B. Memuat TXT
txt_path = "data/demografi-jakarta.txt"
if os.path.exists(txt_path):
    print(f"Memuat dokumen TXT: {txt_path}")
    with open(txt_path, "r", encoding="utf-8") as f:
        text = f.read()
    docs_multi.append(Document(page_content=text, metadata={"source": txt_path}))

print(f"Total halaman/dokumen yang berhasil dimuat: {len(docs_multi)}")

Memuat dokumen PDF: data/pemilu 2024.pdf
Memuat dokumen TXT: data/demografi-jakarta.txt
Total halaman/dokumen yang berhasil dimuat: 3


In [10]:
# 2. Membagi dokumen-dokumen menjadi beberapa chunk/potongan teks kecil
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
)
splits_multi = text_splitter.split_documents(docs_multi)

print(f"Total Chunks : {len(splits_multi)}")

Total Chunks : 8


In [11]:
# 3. Membuat database vektor Chroma (dengan penyimpanan lokal/persist) dan retriever
import shutil
from langchain_chroma import Chroma

persist_directory = "vector_multi"

# Bersihkan folder vector lama jika ingin membuat ulang database vektor dari awal
if os.path.exists(persist_directory):
    shutil.rmtree(persist_directory)

vectorstore_multi = Chroma.from_documents(
    documents=splits_multi,
    embedding=embeddings,
    persist_directory=persist_directory
)

retriever_multi = vectorstore_multi.as_retriever(
    search_kwargs={"k": 4}
)

print("Vector Database Multiple Files Ready (Persisted locally)")

Vector Database Multiple Files Ready (Persisted locally)


In [12]:
# 4. Mendefinisikan tool pencarian multi-dokumen
from langchain_core.tools import tool

@tool
def cari_informasi_multi_dokumen(query: str) -> str:
    """Cari dan dapatkan informasi relevan tentang Pemilu 2024 atau Demografi Jakarta dari dokumen resmi."""
    docs = retriever_multi.invoke(query)
    return "\n\n".join(doc.page_content for doc in docs)

In [13]:
# 5. Membuat Agent untuk RAG Multi-Dokumen
from langchain.agents import create_agent

system_prompt_multi = """
Kamu adalah asisten BPS.
Jawab pertanyaan hanya berdasarkan informasi yang diperoleh dari tool yang tersedia.
Jika jawaban tidak ditemukan, jawab: 'Maaf, saya tidak menemukan informasi tersebut pada dokumen.'
Jangan menggunakan pengetahuan di luar dokumen resmi.
Jawab maksimal 3 kalimat.
"""

agent_multi = create_agent(
    model="google_genai:gemini-3.1-flash-lite",
    tools=[cari_informasi_multi_dokumen],
    system_prompt=system_prompt_multi
)

In [14]:
# 6. Menguji Agent Multi-Dokumen dengan beberapa kueri topik berbeda
from langchain_core.messages import HumanMessage

queries = [
    "kapan pemilu 2024?",
    "berapa persentase suku jawa di jakarta?"
]

for query in queries:
    print(f"=== Pertanyaan: {query} ===")
    question = HumanMessage(content=query)
    response = agent_multi.invoke({"messages": [question]})
    print("Jawaban Agent:")
    print(response["messages"][-1].content)
    print("-" * 50)

=== Pertanyaan: kapan pemilu 2024? ===


Jawaban Agent:
[{'type': 'text', 'text': 'Pemungutan suara Pemilihan Umum (Pemilu) 2024 dilaksanakan pada Rabu, 14 Februari 2024. Sementara itu, pemilihan gubernur, bupati, dan walikota disepakati digelar pada 27 November 2024.', 'extras': {'signature': 'EjQKMgERTTIPPOxifVY4vyJepNRDaixAtL82Hc0EtkQWVu5KxUz9Vs67fHLojHlkWixYM/kj'}}]
--------------------------------------------------
=== Pertanyaan: berapa persentase suku jawa di jakarta? ===


Jawaban Agent:
[{'type': 'text', 'text': 'Berdasarkan data demografi penduduk Jakarta tahun 2024, persentase suku Jawa di Jakarta adalah sebesar 35,16%.', 'extras': {'signature': 'EjQKMgERTTIPzA7Yte/0hAVMQrWC2jo6f+LfPQBIPQknUHrs+3UeKqCrcWwrSQs/RhIfhnqg'}}]
--------------------------------------------------
